# SAE vs CAZ: Side-by-Side Ablation Comparison

Two methods claim to extract interpretable concept directions from a transformer's residual stream:

| | **Sparse Autoencoder (SAE)** | **Concept Assembly Zone (CAZ)** |
|---|---|---|
| **What it learns** | ~24K sparse features via reconstruction loss | 1 direction per concept via contrastive pair geometry |
| **Supervision** | Unsupervised (reconstructs activations) | Weakly supervised (needs contrastive pairs) |
| **Layer choice** | Fixed (e.g. layer 8 residual stream) | Adaptive — extracts at the layer of peak geometric separation |
| **Ablation** | Clamp feature activation to zero, decode back | Orthogonal projection removes direction from hidden state |

This notebook runs **both methods on the same model (GPT-2 Small), same concept, same test prompts** and compares:
1. What each method identifies as the concept's representation
2. How ablation affects the model's internal geometry (separation reduction)
3. How ablation affects the model's behavior (KL divergence, text generation)
4. Whether the two representations agree (geometric alignment)

**Prerequisites:** The [SAE tutorial](./Mechanistic_Interpretability_LLM_Tutorial.ipynb) and [CAZ companion](./CAZ_Companion_Tutorial.ipynb) cover each method in depth. This notebook assumes familiarity with both.

## 0 — Setup

In [ ]:
# Install both toolkits
%pip install -q git+https://github.com/jamesrahenry/Rosetta_Tools.git
%pip install -q sae-lens transformer-lens

In [ ]:
import json
import urllib.request
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# --- SAE stack ---
from sae_lens import SAE
from transformer_lens import HookedTransformer

# --- CAZ stack ---
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
from rosetta_tools.extraction import extract_layer_activations, extract_contrastive_activations
from rosetta_tools.caz import (
    compute_separation, compute_coherence, compute_velocity,
    compute_layer_metrics, find_caz_regions,
)
from rosetta_tools.ablation import (
    DirectionalAblator, get_transformer_layers,
    compute_dominant_direction, compute_baseline_logits,
    kl_divergence_from_logits,
)
from rosetta_tools.gpu_utils import get_device, get_dtype

device = get_device()
dtype = get_dtype(device)
print(f"Device: {device} | dtype: {dtype}")

### Load GPT-2 Small — both views

We load the model twice: once through TransformerLens (for SAE hooks) and once through HuggingFace transformers (for rosetta_tools). Same weights, different wrappers.

In [ ]:
MODEL_NAME = "gpt2"

# --- TransformerLens model (SAE side) ---
tl_model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
print(f"TransformerLens: {MODEL_NAME} — {tl_model.cfg.n_layers}L, d_model={tl_model.cfg.d_model}")

# --- HuggingFace models (CAZ side) ---
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_tokenizer.pad_token = hf_tokenizer.eos_token

# Base model for activation extraction
hf_model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
hf_model.eval()

# CausalLM for logits / text generation
hf_lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
hf_lm.eval()

N_LAYERS = hf_model.config.n_layer
D_MODEL = hf_model.config.n_embd
print(f"HuggingFace: {MODEL_NAME} — {N_LAYERS}L, d_model={D_MODEL}")

### Load SAE dictionary

We use the `gpt2-small-res-jb` SAE trained on layer 8 residual stream pre-activations (Joseph Bloom's dictionary, hosted on HuggingFace via SAELens). 24,576 features learned from the 768-dimensional residual stream — a 32x overcomplete basis.

In [ ]:
SAE_LAYER = 8
SAE_RELEASE = "gpt2-small-res-jb"
SAE_ID = f"blocks.{SAE_LAYER}.hook_resid_pre"
HOOK_POINT = SAE_ID  # TransformerLens hook name

print(f"Loading SAE: {SAE_RELEASE} | {SAE_ID}")
result = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
sae = result[0] if isinstance(result, tuple) else result

print(f"SAE features: {sae.cfg.d_sae}")
print(f"SAE input dim: {sae.cfg.d_in}")

### Load contrastive pairs for the test concept

We'll use **sentiment** — it's a clean, well-separated concept in GPT-2 Small with a clear CAZ peak, and SAE features for positive/negative sentiment are well-documented on Neuronpedia.

In [ ]:
CONCEPT = "sentiment"
N_PAIRS = 200  # cap — will use however many are available

GITHUB_RAW = (
    "https://raw.githubusercontent.com/jamesrahenry/Rosetta_Program/main"
    "/datasets/consensus_pairs"
)

url = f"{GITHUB_RAW}/{CONCEPT}_consensus_pairs.jsonl"
print(f"Fetching: {url}")
with urllib.request.urlopen(url) as resp:
    lines = resp.read().decode().strip().split("\n")
records = [json.loads(line) for line in lines]

pair_map = defaultdict(dict)
for r in records:
    pair_map[r["pair_id"]][r["label"]] = r["text"]

pos_texts, neg_texts = [], []
for pid in sorted(pair_map.keys())[:N_PAIRS]:
    p = pair_map[pid]
    if 1 in p and 0 in p:
        pos_texts.append(p[1])
        neg_texts.append(p[0])

print(f"Loaded {len(pos_texts)} contrastive pairs for '{CONCEPT}'")
print(f"  pos example: {pos_texts[0][:80]}...")
print(f"  neg example: {neg_texts[0][:80]}...")

---
# 1 — Feature Extraction: What Does Each Method Find?

## 1.1 SAE: Sparse feature activations on concept text

In [ ]:
# ============================================================
# Run positive and negative texts through the SAE at layer 8
# ============================================================

def get_sae_features(texts, label=""):
    """Encode texts through TransformerLens -> SAE, return per-text max feature activations."""
    tokens = tl_model.to_tokens(texts).to(device)
    with torch.no_grad():
        _, cache = tl_model.run_with_cache(
            tokens, names_filter=lambda name: name == HOOK_POINT
        )
        x = cache[HOOK_POINT]  # [batch, seq, d_model]

    x_flat = x.reshape(-1, x.shape[-1])
    with torch.no_grad():
        z = sae.encode(x_flat)  # [batch*seq, n_features]

    z = z.reshape(x.shape[0], x.shape[1], -1)
    z_max = z.max(dim=1).values  # [batch, n_features] — max across positions

    if label:
        n_active = (z_max > 0).float().sum(dim=1).mean().item()
        print(f"  {label}: {len(texts)} texts, mean {n_active:.0f} active features per text")

    return z_max

BATCH = 16
pos_feats_list, neg_feats_list = [], []
for i in range(0, len(pos_texts), BATCH):
    pos_feats_list.append(get_sae_features(pos_texts[i:i+BATCH], "pos" if i == 0 else ""))
    neg_feats_list.append(get_sae_features(neg_texts[i:i+BATCH], "neg" if i == 0 else ""))

pos_feats = torch.cat(pos_feats_list, dim=0)  # [n_pos, n_features]
neg_feats = torch.cat(neg_feats_list, dim=0)  # [n_neg, n_features]

In [ ]:
# ============================================================
# Find SAE features that differentiate positive from negative
# ============================================================
# A feature is "concept-discriminative" if it fires much more on one class.
# This is the SAE analogue of difference-of-means.

pos_mean = pos_feats.mean(dim=0)  # [n_features]
neg_mean = neg_feats.mean(dim=0)

# Signed difference: positive = fires more on pos_texts
feat_diff = (pos_mean - neg_mean).cpu().numpy()

# Top features biased toward each class
top_pos_idx = np.argsort(feat_diff)[-10:][::-1]
top_neg_idx = np.argsort(feat_diff)[:10]

print("Top 10 SAE features biased toward POSITIVE sentiment:")
for idx in top_pos_idx:
    print(f"  feature {idx:5d}: pos_mean={pos_mean[idx]:.3f}  neg_mean={neg_mean[idx]:.3f}  diff={feat_diff[idx]:+.3f}")

print(f"\nTop 10 SAE features biased toward NEGATIVE sentiment:")
for idx in top_neg_idx:
    print(f"  feature {idx:5d}: pos_mean={pos_mean[idx]:.3f}  neg_mean={neg_mean[idx]:.3f}  diff={feat_diff[idx]:+.3f}")

# Store the top discriminative features for ablation
N_ABLATE_FEATURES = 10
sae_concept_features = np.concatenate([top_pos_idx[:N_ABLATE_FEATURES//2],
                                        top_neg_idx[:N_ABLATE_FEATURES//2]])
print(f"\nSelected {len(sae_concept_features)} features for ablation: {sae_concept_features.tolist()}")

In [ ]:
# ============================================================
# The SAE "concept direction" — aggregate decoder directions
# ============================================================
# Each SAE feature has a decoder direction W_dec[i] in activation space.
# We compose a single "sentiment direction" by weighting decoder
# directions by their discriminative power (difference of means).

feat_diff_tensor = torch.tensor(feat_diff, device=device, dtype=torch.float32)

# Weight each decoder direction by its signed discriminative score
sae_direction = (feat_diff_tensor[:, None] * sae.W_dec.detach().float()).sum(dim=0)
sae_direction = sae_direction / (sae_direction.norm() + 1e-8)

print(f"SAE composite direction: {sae_direction.shape} (unit vector in d_model={D_MODEL})")
print(f"  Constructed from {(feat_diff_tensor.abs() > 0.01).sum().item():.0f} features with |diff| > 0.01")

## 1.2 CAZ: Layer-wise geometric decomposition

In [ ]:
# ============================================================
# Extract activations at every layer and compute CAZ metrics
# ============================================================

layer_acts = extract_contrastive_activations(
    hf_model, hf_tokenizer, pos_texts, neg_texts, device=device, batch_size=16
)

metrics = compute_layer_metrics(layer_acts)
profile = find_caz_regions(metrics)

print(f"CAZ profile for '{CONCEPT}' in {MODEL_NAME}:")
print(f"  Dominant peak: layer {profile.dominant.peak} ({profile.dominant.depth_pct:.0f}% depth)")
print(f"  Peak separation: {profile.dominant.peak_separation:.3f}")
print(f"  Peak coherence: {profile.dominant.peak_coherence:.3f}")
print(f"  Multimodal: {profile.is_multimodal}")
if profile.is_multimodal:
    for i, r in enumerate(profile.regions):
        print(f"    Region {i}: peak L{r.peak} ({r.depth_pct:.0f}%), sep={r.peak_separation:.3f}")

In [ ]:
# ============================================================
# Extract the DoM direction at the CAZ peak layer
# ============================================================

caz_peak = profile.dominant.peak
pos_acts_peak, neg_acts_peak = layer_acts[caz_peak]
caz_direction_np = compute_dominant_direction(pos_acts_peak, neg_acts_peak)
caz_direction = torch.tensor(caz_direction_np, device=device, dtype=torch.float32)

print(f"CAZ direction: extracted at layer {caz_peak}")
print(f"  Shape: {caz_direction.shape} (unit vector in d_model={D_MODEL})")

In [ ]:
# ============================================================
# Visualize: CAZ metric profile across layers
# ============================================================

layers_range = list(range(len(metrics)))
seps = [m.separation for m in metrics]
cohs = [m.coherence for m in metrics]
vels = [m.velocity for m in metrics]

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

axes[0].plot(layers_range, seps, 'b-o', markersize=3, label='Separation S(l)')
axes[0].axvline(caz_peak, color='red', ls='--', alpha=0.7, label=f'CAZ peak (L{caz_peak})')
axes[0].axvline(SAE_LAYER, color='orange', ls='--', alpha=0.7, label=f'SAE layer (L{SAE_LAYER})')
axes[0].set_ylabel('Separation')
axes[0].legend(fontsize=8)
axes[0].set_title(f'{CONCEPT} — CAZ metrics across depth ({MODEL_NAME})')

axes[1].plot(layers_range, cohs, 'g-o', markersize=3)
axes[1].axvline(caz_peak, color='red', ls='--', alpha=0.7)
axes[1].axvline(SAE_LAYER, color='orange', ls='--', alpha=0.7)
axes[1].set_ylabel('Coherence')

axes[2].plot(layers_range, vels, 'm-o', markersize=3)
axes[2].axvline(caz_peak, color='red', ls='--', alpha=0.7)
axes[2].axvline(SAE_LAYER, color='orange', ls='--', alpha=0.7)
axes[2].axhline(0, color='gray', ls='-', alpha=0.3)
axes[2].set_ylabel('Velocity')
axes[2].set_xlabel('Layer')

plt.tight_layout()
plt.show()

print(f"\nCAZ peak: layer {caz_peak} | SAE dictionary trained at: layer {SAE_LAYER}")
if caz_peak != SAE_LAYER:
    print(f"  These differ by {abs(caz_peak - SAE_LAYER)} layer(s) — the SAE is not extracting at the concept's optimal depth.")
else:
    print(f"  These happen to coincide — the SAE layer matches the concept's peak.")

---
# 2 — Geometric Comparison: Do They Find the Same Thing?

The SAE decomposes activations into ~24K sparse features. The CAZ method extracts a single direction. How much do they overlap?

In [ ]:
# ============================================================
# 2.1 Cosine similarity: SAE composite direction vs CAZ direction
# ============================================================
# Note: the SAE direction is at layer 8, the CAZ direction is at the
# CAZ peak. If they're at different layers, this comparison measures
# agreement in representation space, not identity.

cos_sim = F.cosine_similarity(sae_direction.unsqueeze(0), caz_direction.unsqueeze(0)).item()
print(f"Cosine similarity (SAE composite vs CAZ DoM): {cos_sim:.4f}")
print()
if abs(cos_sim) > 0.7:
    print("Strong alignment — both methods are finding a similar direction.")
elif abs(cos_sim) > 0.3:
    print("Moderate alignment — partial overlap but not the same direction.")
else:
    print("Weak alignment — the methods are finding different structures.")
if caz_peak != SAE_LAYER:
    print(f"  (They operate at different layers — L{SAE_LAYER} vs L{caz_peak})")

In [ ]:
# ============================================================
# 2.2 Per-feature alignment: which SAE features point along the CAZ direction?
# ============================================================

W_dec = sae.W_dec.detach().float()  # [n_features, d_model]
projections = (W_dec @ caz_direction).cpu().numpy()  # [n_features]

top_aligned = np.argsort(np.abs(projections))[-20:][::-1]

print("Top 20 SAE features most aligned with the CAZ sentiment direction:")
print(f"{'Feature':>8s}  {'Projection':>10s}  {'|Proj|':>8s}  {'Concept-discrim':>15s}")
for idx in top_aligned:
    print(f"{idx:8d}  {projections[idx]:+10.4f}  {abs(projections[idx]):8.4f}  {feat_diff[idx]:+15.4f}")

# Are the concept-discriminative features also CAZ-aligned?
discrim_features = np.concatenate([top_pos_idx[:5], top_neg_idx[:5]])
print(f"\nCAZ-alignment of the top concept-discriminative features:")
for idx in discrim_features:
    rank = np.where(np.argsort(np.abs(projections))[::-1] == idx)[0][0]
    print(f"  feature {idx:5d}: projection={projections[idx]:+.4f}, rank={rank+1}/{len(projections)}")

In [ ]:
# ============================================================
# 2.3 Visualize: projection distribution
# ============================================================

fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(projections, bins=100, alpha=0.6, color='steelblue', label='All SAE features')

for idx in top_pos_idx[:5]:
    ax.axvline(projections[idx], color='green', ls='--', alpha=0.7,
               label='Top pos-sentiment' if idx == top_pos_idx[0] else None)
for idx in top_neg_idx[:5]:
    ax.axvline(projections[idx], color='red', ls='--', alpha=0.7,
               label='Top neg-sentiment' if idx == top_neg_idx[0] else None)

ax.set_xlabel('Projection onto CAZ direction')
ax.set_ylabel('Count')
ax.set_title('SAE feature decoder directions projected onto CAZ sentiment direction')
ax.legend()
plt.tight_layout()
plt.show()

total_proj_energy = (projections ** 2).sum()
top10_energy = sum(projections[i]**2 for i in top_aligned[:10])
print(f"\nFraction of CAZ direction energy captured by top 10 SAE features: {top10_energy/total_proj_energy:.1%}")
print(f"Fraction captured by top 50: {sum(projections[i]**2 for i in top_aligned[:50])/total_proj_energy:.1%}")

---
# 3 — Ablation: Removing the Concept

Now the real test. Both methods claim to have found the concept's representation. If we remove it, does the model lose the ability to distinguish high-sentiment from low-sentiment inputs?

## 3.1 CAZ ablation: orthogonal projection at the peak layer

For each hidden state $h$ and concept direction $\hat{v}$, the ablation computes $h' = h - (h \cdot \hat{v})\hat{v}$ — the component along the concept direction is removed exactly. No reconstruction error, no dark matter.

In [ ]:
# ============================================================
# CAZ ablation: remove the DoM direction at the CAZ peak
# ============================================================
# Ablation at layer L modifies that layer's output, so its effect
# is visible at layer L+1 onward. We measure one layer downstream.

layers = get_transformer_layers(hf_model)
measure_layer = min(caz_peak + 1, N_LAYERS)

pos_acts_m, neg_acts_m = layer_acts[measure_layer]
baseline_sep_caz = compute_separation(pos_acts_m, neg_acts_m)
print(f"Baseline separation at L{measure_layer} (downstream of peak L{caz_peak}): {baseline_sep_caz:.4f}")

with DirectionalAblator(layers[caz_peak], caz_direction_np, dtype=dtype):
    ablated_acts = extract_contrastive_activations(
        hf_model, hf_tokenizer, pos_texts, neg_texts, device=device, batch_size=16
    )

ablated_pos, ablated_neg = ablated_acts[measure_layer]
ablated_sep_caz = compute_separation(ablated_pos, ablated_neg)
caz_reduction = 1.0 - ablated_sep_caz / baseline_sep_caz

print(f"Ablated separation at L{measure_layer}: {ablated_sep_caz:.4f}")
print(f"Separation reduction: {caz_reduction:.1%}")

## 3.2 SAE ablation: clamp concept features to zero

The SAE approach: encode activations into sparse features, zero out the concept-relevant ones, decode back. This replaces the *entire* activation with the SAE reconstruction (modified). The reconstruction error ("dark matter") is injected even when no features are ablated — it's the cost of the encode-decode roundtrip.

In [ ]:
# ============================================================
# SAE ablation: encode -> zero features -> decode
# Measured at the final layer (downstream behavioral effect)
# ============================================================

def measure_separation_sae_ablated(pos_texts, neg_texts, features_to_ablate, batch_size=8):
    all_pos_acts, all_neg_acts = [], []

    for texts, collector in [(pos_texts, all_pos_acts), (neg_texts, all_neg_acts)]:
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            tokens = tl_model.to_tokens(batch).to(device)

            def make_hook(feats):
                def ablation_hook(act, hook):
                    shape = act.shape
                    act_flat = act.reshape(-1, shape[-1])
                    with torch.no_grad():
                        z = sae.encode(act_flat)
                        for fi in feats:
                            z[:, fi] = 0.0
                        act_recon = sae.decode(z)
                    return act_recon.reshape(shape)
                return ablation_hook

            hook_fn = make_hook(features_to_ablate)
            final_hook_name = f"blocks.{N_LAYERS-1}.hook_resid_post"

            with torch.no_grad():
                with tl_model.hooks(fwd_hooks=[(HOOK_POINT, hook_fn)]):
                    _, cache = tl_model.run_with_cache(
                        tokens,
                        names_filter=lambda name: name == final_hook_name,
                    )
                final_acts = cache[final_hook_name][:, -1, :]
                collector.append(final_acts.cpu().float().numpy())

    pos_arr = np.concatenate(all_pos_acts, axis=0)
    neg_arr = np.concatenate(all_neg_acts, axis=0)
    return compute_separation(pos_arr, neg_arr)

# Baseline: SAE reconstruct with NO features zeroed (measures dark matter cost)
baseline_sep_sae = measure_separation_sae_ablated(pos_texts, neg_texts, features_to_ablate=[])
print(f"Baseline separation (SAE reconstruct, no ablation): {baseline_sep_sae:.4f}")
print(f"  Compare: real baseline (no SAE) at L{measure_layer}: {baseline_sep_caz:.4f}")
print(f"  Dark matter cost: {1 - baseline_sep_sae/baseline_sep_caz:.1%} of concept signal lost to reconstruction error")

# Ablate top concept features
ablated_sep_sae = measure_separation_sae_ablated(pos_texts, neg_texts, sae_concept_features)
sae_reduction = 1.0 - ablated_sep_sae / baseline_sep_sae
print(f"\nAblated separation ({len(sae_concept_features)} features zeroed): {ablated_sep_sae:.4f}")
print(f"Separation reduction (vs SAE baseline): {sae_reduction:.1%}")

## 3.3 Fair comparison: SAE direction via orthogonal projection

The SAE feature clamping result is confounded by dark matter — the reconstruction error dominates the signal. For a fair comparison, we use the **same ablation mechanism** (orthogonal projection) with the **SAE's composite direction** instead of the CAZ direction. This isolates the question: *does the SAE find a good direction?* from *does the SAE reconstruction lose signal?*

In [ ]:
# ============================================================
# Orthogonal projection ablation using the SAE composite direction
# Same mechanism as CAZ, different direction, different layer
# ============================================================

sae_direction_np = sae_direction.cpu().numpy().astype(np.float64)
sae_direction_np = sae_direction_np / (np.linalg.norm(sae_direction_np) + 1e-8)

sae_measure_layer = min(SAE_LAYER + 1, N_LAYERS)
pos_acts_sae_m, neg_acts_sae_m = layer_acts[sae_measure_layer]
baseline_sep_sae_dir = compute_separation(pos_acts_sae_m, neg_acts_sae_m)
print(f"Baseline separation at L{sae_measure_layer} (downstream of SAE L{SAE_LAYER}): {baseline_sep_sae_dir:.4f}")

with DirectionalAblator(layers[SAE_LAYER], sae_direction_np, dtype=dtype):
    sae_dir_acts = extract_contrastive_activations(
        hf_model, hf_tokenizer, pos_texts, neg_texts, device=device, batch_size=16
    )

abl_pos_sae, abl_neg_sae = sae_dir_acts[sae_measure_layer]
ablated_sep_sae_dir = compute_separation(abl_pos_sae, abl_neg_sae)
sae_dir_reduction = 1.0 - ablated_sep_sae_dir / baseline_sep_sae_dir

print(f"Ablated separation (SAE direction, ortho projection at L{SAE_LAYER}): {ablated_sep_sae_dir:.4f}")
print(f"Separation reduction: {sae_dir_reduction:.1%}")
print(f"\nFair comparison (same mechanism, different directions):")
print(f"  CAZ direction at L{caz_peak}: {caz_reduction:.1%} separation reduction")
print(f"  SAE direction at L{SAE_LAYER}: {sae_dir_reduction:.1%} separation reduction")

## 3.4 Behavioral impact: KL divergence on general prompts

In [ ]:
# ============================================================
# Prompts for behavioral measurement
# ============================================================

CAPABILITY_PROMPTS = [
    "The capital of France is",
    "Water boils at a temperature of",
    "The mitochondria is the powerhouse of the",
    "In 1969, humans first landed on the",
    "The speed of light is approximately",
    "The largest planet in the solar system is",
    "Photosynthesis converts sunlight into",
    "The Pythagorean theorem states that",
]

SENTIMENT_PROMPTS = [
    "The restaurant was absolutely",
    "I felt deeply disappointed because",
    "The customer service was incredibly",
    "This product exceeded all my",
    "The experience was utterly",
    "I'm thrilled to report that",
    "The worst part about this was",
    "I was pleasantly surprised by the",
]

In [ ]:
# ============================================================
# CAZ ablation: KL divergence
# ============================================================

baseline_logits_cap = compute_baseline_logits(hf_lm, hf_tokenizer, CAPABILITY_PROMPTS, device)
baseline_logits_sent = compute_baseline_logits(hf_lm, hf_tokenizer, SENTIMENT_PROMPTS, device)

lm_layers = get_transformer_layers(hf_lm)

with DirectionalAblator(lm_layers[caz_peak], caz_direction_np, dtype=dtype):
    ablated_logits_cap = compute_baseline_logits(hf_lm, hf_tokenizer, CAPABILITY_PROMPTS, device)
    ablated_logits_sent = compute_baseline_logits(hf_lm, hf_tokenizer, SENTIMENT_PROMPTS, device)

caz_kl_cap = np.mean([
    kl_divergence_from_logits(b, a)
    for b, a in zip(baseline_logits_cap, ablated_logits_cap)
])
caz_kl_sent = np.mean([
    kl_divergence_from_logits(b, a)
    for b, a in zip(baseline_logits_sent, ablated_logits_sent)
])

print(f"CAZ ablation (L{caz_peak}):")
print(f"  KL on capability prompts:  {caz_kl_cap:.4f}  (collateral damage)")
print(f"  KL on sentiment prompts:   {caz_kl_sent:.4f}  (intended effect)")
print(f"  Selectivity ratio:         {caz_kl_sent / (caz_kl_cap + 1e-8):.1f}x")

In [ ]:
# ============================================================
# SAE ablation: KL divergence (feature clamping)
# ============================================================

def compute_kl_sae_ablated(prompts, features_to_ablate):
    tokens = tl_model.to_tokens(prompts).to(device)

    with torch.no_grad():
        orig_logits = tl_model(tokens)
    orig_probs = torch.softmax(orig_logits[:, -1, :], dim=-1)

    def make_hook(feats):
        def hook_fn(act, hook):
            shape = act.shape
            act_flat = act.reshape(-1, shape[-1])
            with torch.no_grad():
                z = sae.encode(act_flat)
                for fi in feats:
                    z[:, fi] = 0.0
                return sae.decode(z).reshape(shape)
        return hook_fn

    with torch.no_grad():
        ablated_logits = tl_model.run_with_hooks(
            tokens, fwd_hooks=[(HOOK_POINT, make_hook(features_to_ablate))]
        )
    ablated_probs = torch.softmax(ablated_logits[:, -1, :], dim=-1)

    kl = F.kl_div(
        ablated_probs.log(), orig_probs, reduction="none", log_target=False
    ).sum(dim=-1).mean().item()
    return kl

sae_kl_cap = compute_kl_sae_ablated(CAPABILITY_PROMPTS, sae_concept_features)
sae_kl_sent = compute_kl_sae_ablated(SENTIMENT_PROMPTS, sae_concept_features)

print(f"SAE ablation ({len(sae_concept_features)} features):")
print(f"  KL on capability prompts:  {sae_kl_cap:.4f}  (collateral damage)")
print(f"  KL on sentiment prompts:   {sae_kl_sent:.4f}  (intended effect)")
print(f"  Selectivity ratio:         {sae_kl_sent / (sae_kl_cap + 1e-8):.1f}x")

## 3.5 Text generation comparison

In [ ]:
# ============================================================
# Generate text: baseline, CAZ-ablated, SAE-ablated
# ============================================================

GEN_PROMPTS = [
    "The restaurant was absolutely",
    "I felt deeply disappointed because",
    "This product exceeded all my",
]

def generate_hf(prompt, max_new_tokens=40, seed=42):
    torch.manual_seed(seed)
    enc = hf_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = hf_lm.generate(
            **enc, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.8, top_p=0.95,
            pad_token_id=hf_tokenizer.eos_token_id,
        )
    return hf_tokenizer.decode(out[0], skip_special_tokens=True)

def generate_tl(prompt, max_new_tokens=40, seed=42, fwd_hooks=None):
    torch.manual_seed(seed)
    if fwd_hooks:
        with tl_model.hooks(fwd_hooks=fwd_hooks):
            return tl_model.generate(
                prompt, max_new_tokens=max_new_tokens,
                temperature=0.8, do_sample=True, top_p=0.95, verbose=False,
            )
    return tl_model.generate(
        prompt, max_new_tokens=max_new_tokens,
        temperature=0.8, do_sample=True, top_p=0.95, verbose=False,
    )

def make_sae_ablation_hook(features):
    def hook(act, hook_obj):
        shape = act.shape
        act_flat = act.reshape(-1, shape[-1])
        with torch.no_grad():
            z = sae.encode(act_flat)
            for fi in features:
                z[:, fi] = 0.0
            return sae.decode(z).reshape(shape)
    return hook

print("=" * 80)
for prompt in GEN_PROMPTS:
    print(f"\nPrompt: \"{prompt}\"\n")

    baseline = generate_hf(prompt)
    print(f"  Baseline:     {baseline}")

    with DirectionalAblator(lm_layers[caz_peak], caz_direction_np, dtype=dtype):
        caz_gen = generate_hf(prompt)
    print(f"  CAZ ablated:  {caz_gen}")

    sae_gen = generate_tl(
        prompt, fwd_hooks=[(HOOK_POINT, make_sae_ablation_hook(sae_concept_features))]
    )
    print(f"  SAE ablated:  {sae_gen}")

    print("-" * 80)

---
# 4 — Summary Comparison

In [ ]:
# ============================================================
# Summary table
# ============================================================

summary = pd.DataFrame([
    {
        "Method": "SAE feature clamping",
        "Mechanism": "encode->zero->decode",
        "Layer": SAE_LAYER,
        "# Features/Dirs": len(sae_concept_features),
        "Sep Reduction": f"{sae_reduction:.1%}",
        "KL (sent)": f"{sae_kl_sent:.4f}",
        "KL (cap)": f"{sae_kl_cap:.4f}",
    },
    {
        "Method": "SAE direction (fair)",
        "Mechanism": "ortho projection",
        "Layer": SAE_LAYER,
        "# Features/Dirs": 1,
        "Sep Reduction": f"{sae_dir_reduction:.1%}",
        "KL (sent)": "—",
        "KL (cap)": "—",
    },
    {
        "Method": "CAZ directional ablation",
        "Mechanism": "ortho projection",
        "Layer": caz_peak,
        "# Features/Dirs": 1,
        "Sep Reduction": f"{caz_reduction:.1%}",
        "KL (sent)": f"{caz_kl_sent:.4f}",
        "KL (cap)": f"{caz_kl_cap:.4f}",
    },
])

print(summary.to_string(index=False))

In [ ]:
# ============================================================
# Visualization: CAZ layer sweep — which layer matters most?
# ============================================================

layer_reductions = []
for li in range(N_LAYERS):
    pos_li, neg_li = layer_acts[li]
    direction_li = compute_dominant_direction(pos_li, neg_li)

    m_layer = min(li + 1, N_LAYERS)
    pos_m, neg_m = layer_acts[m_layer]
    base = compute_separation(pos_m, neg_m)

    with DirectionalAblator(layers[li], direction_li, dtype=dtype):
        abl = extract_contrastive_activations(
            hf_model, hf_tokenizer, pos_texts, neg_texts, device=device, batch_size=16
        )
    abl_p, abl_n = abl[m_layer]
    abl_sep = compute_separation(abl_p, abl_n)
    layer_reductions.append(1.0 - abl_sep / base if base > 0 else 0.0)
    print(f"  L{li:2d}: {layer_reductions[-1]:.1%}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(N_LAYERS), [r * 100 for r in layer_reductions], color='steelblue', alpha=0.7)
ax.axvline(caz_peak, color='red', ls='--', label=f'CAZ peak (L{caz_peak})')
ax.axvline(SAE_LAYER, color='orange', ls='--', label=f'SAE layer (L{SAE_LAYER})')
ax.set_xlabel('Ablation layer')
ax.set_ylabel('Separation reduction (%)')
ax.set_title(f'Which layer matters most for {CONCEPT}? (directional ablation)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Dark matter comparison
# ============================================================

sample_texts = pos_texts[:32] + neg_texts[:32]
tokens = tl_model.to_tokens(sample_texts).to(device)

with torch.no_grad():
    _, cache = tl_model.run_with_cache(
        tokens, names_filter=lambda name: name == HOOK_POINT
    )
    x = cache[HOOK_POINT]

x_flat = x.reshape(-1, x.shape[-1])
with torch.no_grad():
    z = sae.encode(x_flat)
    x_hat = sae.decode(z)

err = (x_flat - x_hat).float()
x_float = x_flat.float()

explained_var = 1.0 - (err.pow(2).sum(dim=-1) / (x_float.pow(2).sum(dim=-1) + 1e-8))

print("SAE reconstruction at layer 8:")
print(f"  Mean explained variance: {explained_var.mean().item():.4f}")
print(f"  Dark matter: {(1 - explained_var.mean().item())*100:.1f}% of activation variance unexplained")

# CAZ: variance removed by ablation = fraction of total variance in the concept direction
sample_pos = torch.tensor(pos_acts_peak[:32], device=device, dtype=torch.float32)
sample_neg = torch.tensor(neg_acts_peak[:32], device=device, dtype=torch.float32)
sample_all = torch.cat([sample_pos, sample_neg], dim=0)

caz_dir_t = caz_direction.unsqueeze(0)
projections_onto_caz = (sample_all @ caz_dir_t.T) * caz_dir_t
caz_var_removed = projections_onto_caz.pow(2).sum() / sample_all.pow(2).sum()

print(f"\nCAZ ablation (1 direction at L{caz_peak}):")
print(f"  Variance removed: {caz_var_removed.item()*100:.1f}%")
print(f"  Variance retained: {(1 - caz_var_removed.item())*100:.1f}%")
print(f"\nThe SAE touches {(1 - explained_var.mean().item())*100:.1f}% of variance as dark matter")
print(f"while CAZ removes only {caz_var_removed.item()*100:.1f}% — a surgical cut vs. a lossy reconstruction.")

---
# 5 — Discussion: When to Use Which

### What this comparison shows

**The SAE's dark matter problem is severe for concept-level ablation.** The encode-decode roundtrip destroys most concept geometry before any features are ablated. This isn't a bug in SAEs — they're optimized for reconstruction fidelity across *all* concepts, not for preserving any single concept's discriminative structure.

**Layer choice matters as much as direction quality.** The fair comparison (Section 3.3) uses the same ablation mechanism but different directions at different layers. The CAZ advantage comes from finding the concept's *optimal* layer, not just a better direction.

**The methods are complementary, not competing:**

| Criterion | SAE | CAZ |
|---|---|---|
| **Granularity** | Individual features — can identify *which specific aspect* of sentiment (word choice, intensifiers, hedging) | Single direction — captures the concept as a whole |
| **Layer flexibility** | Fixed to the layer the SAE was trained on | Adapts to each concept's peak layer |
| **Supervision** | None (unsupervised reconstruction) | Needs contrastive pairs (weak supervision) |
| **Ablation precision** | Can target individual sub-features | Removes the entire concept direction cleanly |
| **Collateral damage** | SAE reconstruction introduces dark matter error even with no ablation | Orthogonal projection is exact — zero error on non-concept dimensions |
| **Scalability** | One SAE per layer per model; training is expensive | Direction extraction is one forward pass; no training |
| **Interpretability** | Each feature has a Neuronpedia page with activation examples | One direction — interpretable via the contrastive pair semantics |
| **Monitoring use case** | Feature dashboards, anomaly detection on feature distributions | Real-time concept scoring (the CIA use case) |

- Use **SAEs** when you want to understand *what specific features* compose a concept, explore the feature space, or build dashboards.
- Use **CAZ** when you want to *score a concept's presence* in a single number, find the optimal layer to intervene, or monitor concept suppression in real time.

The geometric alignment analysis (Section 2) shows how they connect: the CAZ direction is approximately a weighted sum of SAE features, and the most concept-discriminative SAE features tend to align with the CAZ direction — but imperfectly, because the SAE distributes the concept across many features rather than concentrating it.

In [ ]:
print("Notebook complete.")
print(f"Model: {MODEL_NAME}")
print(f"Concept: {CONCEPT}")
print(f"SAE layer: {SAE_LAYER} | CAZ peak: {caz_peak}")
print(f"Cosine similarity (SAE composite <> CAZ direction): {cos_sim:.4f}")
print(f"\nSeparation reduction:")
print(f"  SAE feature clamping:  {sae_reduction:.1%} (confounded by dark matter)")
print(f"  SAE direction (fair):  {sae_dir_reduction:.1%}")
print(f"  CAZ direction:         {caz_reduction:.1%}")